# **1. Setup & Libraries**

In [ ]:
!pip install matplotlib-scalebar
!pip install contextily

# For post-hoc seasonal tests
!pip install scikit-posthocs

!pip install pymannkendall


In [ ]:
!pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 4.2 MB/s eta 0:00:24
    --------------------------------------- 1.3/100.2 MB 4.0 MB/s eta 0:00:26
    --------------------------------------- 1.8/100.2 MB 3.2 MB/s eta 0:00:31
   - -------------------------------------- 2.6/100.2 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 3.1/100.2 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 4.2/100.2 MB 3.5 MB/s eta 0:00:28
   - -------------------------------------- 5.0/100.2 MB 3.6 MB/s eta 0:00:27
   -- ------------------------------------- 5.8/100.2 MB 3.7 MB/s eta 0:00:26
   -- ------------------------------------- 6.8/100.2 MB 3.8 MB/s eta 0:00:25
   --- ------------------------------------ 7.9/100.2 MB 4.0 MB/s eta 0:00:24
   --- ------------------------------------ 8.4/100.2 MB 3.8 MB/s eta 0:00:25
   --- ------------------------------------ 8.9/100.2 MB 3.8 MB/s eta 0

In [ ]:
!pip install matplotlib-scalebar

In [ ]:
!pip install lightgbm


In [ ]:
import os
import glob
import calendar

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import ScalarFormatter
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, wilcoxon
import pymannkendall as mk
import scikit_posthocs as sp


# **2. Load Dataset**

In [ ]:
file_path = r"C:\Users\deepn\Documents\Rainfall_Training_Research_Lab\Data\Final processed 3-hourly rainfall data with missing value.csv"
df_rain = pd.read_csv(file_path)
df_rain

,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2121051,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 09:00:00,2023,12,31,9,Winter,14.0,NaN,NaN,NaN,50.2,0.0
2121052,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 12:00:00,2023,12,31,12,Winter,15.7,NaN,NaN,NaN,71.5,0.0
2121053,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 15:00:00,2023,12,31,15,Winter,15.6,NaN,NaN,NaN,78.4,0.0
2121054,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 18:00:00,2023,12,31,18,Winter,14.8,NaN,NaN,NaN,84.2,0.0


In [ ]:
# REMOVE THE LATE STATION (Example ID: 10325)
# Replace 10325 with the actual ID of the 2008 station
stations_to_drop = [11921]
df_rain = df_rain[~df_rain["StationID"].isin(stations_to_drop)].reset_index(drop=True)
df_rain


,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2074299,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 09:00:00,2023,12,31,9,Winter,14.0,NaN,NaN,NaN,50.2,0.0
2074300,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 12:00:00,2023,12,31,12,Winter,15.7,NaN,NaN,NaN,71.5,0.0
2074301,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 15:00:00,2023,12,31,15,Winter,15.6,NaN,NaN,NaN,78.4,0.0
2074302,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-12-31 18:00:00,2023,12,31,18,Winter,14.8,NaN,NaN,NaN,84.2,0.0


# **Hyperparameter Training**

In [ ]:
!pip install quantile-forest

   ---------------------------------------- 0.0/685.6 kB ? eta -:--:--
   --------------- ------------------------ 262.1/685.6 kB ? eta -:--:--
   ---------------------------------------- 685.6/685.6 kB 3.4 MB/s  0:00:00


In [ ]:
# ============================================================
# STATION-WISE QUANTILE RANDOM FOREST BASELINE - HYPERPARAMETER TUNING
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import joblib

from quantile_forest import RandomForestQuantileRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TUNING_SAVE_DIR = r"/content/drive/MyDrive/Rainfall/Updated Research Data/Baselines/QRF/Tuning_v1"
os.makedirs(TUNING_SAVE_DIR, exist_ok=True)

# ============================================================
# 1) Core utilities
# ============================================================
class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8): self.eps = eps
    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.where(np.nanstd(X, axis=0) < self.eps, 1.0, np.nanstd(X, axis=0))
        return self
    def transform(self, X): return (X - self.mean_) / self.std_

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None: return (loss * mask).sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    return 2.0 * torch.stack([pinball_loss(y_hat[..., k], y_true, q, mask=mask) for k, q in enumerate(quantiles)]).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None: return (loss * mask).sum() / mask.sum().clamp_min(1.0)
    return loss.mean()

def safe_div(a, b, eps=1e-12): return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50: return {"n_valid": int(valid.sum()), "pr_auc": np.nan, "roc_auc": np.nan, "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds}}
    p, y = probs[valid], y_true[valid]
    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try: roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except: roc_auc = np.nan
    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP, FP, FN = ((yhat == 1) & (y == 1)).sum(), ((yhat == 1) & (y == 0)).sum(), ((yhat == 0) & (y == 1)).sum()
        by_thr[thr] = {"POD": float(safe_div(TP, TP + FN)), "FAR": float(safe_div(FP, TP + FP)), "CSI": float(safe_div(TP, TP + FP + FN))}
    return {"n_valid": int(valid.sum()), "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan, "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan, "by_thr": by_thr}

def make_splits(T, train_frac=0.7, val_frac=0.15): return int(T * train_frac), int(T * (train_frac + val_frac))

def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    scaler = NaNIgnoringStandardScaler().fit(X_flat[:train_end*N])
    return scaler.transform(X_flat).reshape(T, N, F).astype(np.float32), scaler

def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train, M_train, N = Y[:train_end], M[:train_end], Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)
    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0
    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        thr[j] = np.nanpercentile(vals, q*100) if len(vals) >= 100 else global_fallback
    return thr

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc, Mask = np.full((T, N), np.nan, dtype=np.float32), np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window, wmask = Y_rain[t+1:t+1+H_out], M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok], Acc[t, ok] = 1.0, window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing & Dataset
# ============================================================
def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)
    dr_rad = np.deg2rad(df["DR"].astype(np.float32).to_numpy())
    df["DR_sin"], df["DR_cos"] = np.sin(dr_rad), np.cos(dr_rad)
    stations = df[["StationID", "StationName", "Latitude", "Longitude"]].drop_duplicates().sort_values("StationID").reset_index(drop=True)
    stations["node_index"] = np.arange(len(stations))
    N, times = len(stations), np.sort(df["Datetime"].unique())
    T = len(times)

    full_index = pd.MultiIndex.from_product([times, stations["StationID"].values], names=["Datetime", "StationID"])
    meteo_cols = ["DewPointTemperature", "StationLevelPressure", "SP", "Humidity", "Rainfall", "DR_sin", "DR_cos"]
    X_raw = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index().reindex(full_index).values.reshape(T, N, len(meteo_cols)).astype(np.float32)
    M_in = (~np.isnan(X_raw)).astype(np.float32)
    Y_rain, M_y = X_raw[..., meteo_cols.index("Rainfall")], M_in[..., meteo_cols.index("Rainfall")]

    dt_index = pd.DatetimeIndex(times)
    hour, month = dt_index.hour.astype(np.float32), dt_index.month.astype(np.float32)
    time_feats = np.repeat(np.stack([np.sin(2*np.pi*(hour/24.0)), np.cos(2*np.pi*(hour/24.0)), np.sin(2*np.pi*((month-1)/12.0)), np.cos(2*np.pi*((month-1)/12.0))], axis=-1).astype(np.float32)[:, None, :], N, axis=1)
    season_by_time = df.groupby("Datetime")["Season"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]).reindex(times).astype(str).values
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}

    return {"stations": stations, "times": times, "X_raw": X_raw, "M_in": M_in, "Y_rain": Y_rain, "M_y": M_y, "meteo_cols": meteo_cols, "time_feats": time_feats, "season_ids": np.array([season_to_id[s] for s in season_by_time], dtype=np.int64), "unique_seasons": unique_seasons}

class BanglaRainDataset(Dataset):
    def __init__(self, X_scaled, M_in, time_feats, Y_rain, M_y, Acc24, MaskAcc24, season_ids, thr3h, thrAcc24, t_start, t_end, T_in=16, H_out=8, peak_min_obs=None):
        self.X_scaled, self.M_in, self.time_feats = X_scaled, M_in, time_feats
        self.Y_rain, self.M_y, self.Acc24, self.MaskAcc24 = Y_rain, M_y, Acc24, MaskAcc24
        self.season_ids, self.thr3h, self.thrAcc24, self.T_in, self.H_out = season_ids, thr3h, thrAcc24, T_in, H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)
        self.indices = list(range(t_start + (T_in - 1), t_end - H_out))

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        t = self.indices[idx]
        x_all = np.concatenate([np.nan_to_num(self.X_scaled[t-self.T_in+1:t+1], nan=-999.0).astype(np.float32), self.M_in[t-self.T_in+1:t+1].astype(np.float32), self.time_feats[t-self.T_in+1:t+1]], axis=-1)
        y_win, m_win = self.Y_rain[t+1:t+1+self.H_out], self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y_win, nan=0.0)).astype(np.float32)
        m_next = self.M_y[t+1].astype(np.float32)
        flash = ((self.Y_rain[t+1] >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)

        # Suppress the All-NaN slice warning for max peaking
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)

        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((self.Acc24[t] >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)
        return torch.from_numpy(x_all), torch.tensor(int(self.season_ids[t]), dtype=torch.long), torch.from_numpy(y_log), torch.from_numpy(m_win), torch.from_numpy(flash), torch.from_numpy(m_next), torch.from_numpy(peak24), torch.from_numpy(mpeak), torch.from_numpy(acc24), torch.from_numpy(macc)

# ============================================================
# 3) Build Tabular
# ============================================================
def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac, val_frac)
    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)
    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)
    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, 0, train_end, T_in, H_out)
    ds_val = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, train_end, val_end, T_in, H_out)
    ds_test = BanglaRainDataset(X_scaled, prep["M_in"], prep["time_feats"], prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24, prep["season_ids"], thr3h, thrAcc24, val_end, T, T_in, H_out)
    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)

def make_loaders(ds_train, ds_val, ds_test, batch_size=256):
    return DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True), DataLoader(ds_val, batch_size=batch_size), DataLoader(ds_test, batch_size=batch_size)

def build_tabular_from_loader(loader):
    X_reg_list, y_reg_list, station_reg_list, lead_reg_list, mask_reg_list = [], [], [], [], []
    X_clf_list, station_clf_list, flash_y_list, flash_m_list, peak_y_list, peak_m_list, acc_y_list, acc_m_list = [], [], [], [], [], [], [], []

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x_np = x.numpy()
        B, T_in, N, F_all = x_np.shape
        x_flat = np.transpose(x_np, (0, 2, 1, 3)).reshape(B, N, T_in * F_all)
        y_log_np, my_np = y_log.numpy(), my.numpy()
        flash_np, mflash_np, peak_np, mpeak_np, acc_np, macc_np = flash.numpy(), mflash.numpy(), peak.numpy(), mpeak.numpy(), acc.numpy(), macc.numpy()
        B, H, N = y_log_np.shape

        for b in range(B):
            for h in range(H):
                mask_h = my_np[b, h, :] > 0.5
                if not mask_h.any(): continue
                X_bh, y_bh = x_flat[b], y_log_np[b, h, :]
                for j in range(N):
                    if mask_h[j]:
                        X_reg_list.append(X_bh[j]); y_reg_list.append(y_bh[j]); station_reg_list.append(j); lead_reg_list.append(h); mask_reg_list.append(1.0)
            for j in range(N):
                if mflash_np[b, j] > 0.5:
                    X_clf_list.append(x_flat[b, j]); station_clf_list.append(j)
                    flash_y_list.append(flash_np[b, j]); peak_y_list.append(peak_np[b, j]); acc_y_list.append(acc_np[b, j])
                    flash_m_list.append(mflash_np[b, j]); peak_m_list.append(mpeak_np[b, j]); acc_m_list.append(macc_np[b, j])

    return {
        "X_reg": np.array(X_reg_list, dtype=np.float32), "y_reg": np.array(y_reg_list, dtype=np.float32), "station_reg": np.array(station_reg_list, dtype=np.int64), "lead_reg": np.array(lead_reg_list, dtype=np.int64), "mask_reg": np.array(mask_reg_list, dtype=np.float32),
        "X_clf": np.array(X_clf_list, dtype=np.float32), "station_clf": np.array(station_clf_list, dtype=np.int64),
        "flash_y": np.array(flash_y_list, dtype=np.float32), "flash_m": np.array(flash_m_list, dtype=np.float32),
        "peak_y": np.array(peak_y_list, dtype=np.float32), "peak_m": np.array(peak_m_list, dtype=np.float32),
        "acc_y": np.array(acc_y_list, dtype=np.float32), "acc_m": np.array(acc_m_list, dtype=np.float32),
    }

# ============================================================
# 4) Model Wrapper & Evaluate
# ============================================================
class StationWiseQRFBaseline(nn.Module):
    def __init__(self, qrf_regressors, qrf_flash, qrf_peak, qrf_acc, num_stations, H_out, quantiles):
        super().__init__()
        self.qrf_regressors, self.qrf_flash, self.qrf_peak, self.qrf_acc = qrf_regressors, qrf_flash, qrf_peak, qrf_acc
        self.N, self.H_out, self.quantiles = num_stations, H_out, list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T_in, N, F_all = x.shape
        x_flat = np.transpose(x.detach().cpu().numpy(), (0, 2, 1, 3)).reshape(B, N, T_in * F_all)
        q_log, flash_prob, peak_prob, acc_prob = np.zeros((B, self.H_out, N, self.K), dtype=np.float32), np.zeros((B, N), dtype=np.float32), np.zeros((B, N), dtype=np.float32), np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_flat[:, j, :]
            for h in range(self.H_out):
                reg_model = self.qrf_regressors.get((j, h), None)
                if reg_model is None: q_log[:, h, j, :] = 0.0
                else:
                    yhat = reg_model.predict(X_j, quantiles=self.quantiles)
                    q_log[:, h, j, :] = yhat[:, None] if yhat.ndim == 1 else yhat

            def _get_probs(model, X_sub):
                if model is None: return np.zeros(B, dtype=np.float32)
                p = model.predict_proba(X_sub)
                return p[:, 1] if p.shape[1] == 2 else (p[:, 0] if (hasattr(model, 'classes_') and model.classes_[0] == 1) else np.zeros(B, dtype=np.float32))

            flash_prob[:, j] = _get_probs(self.qrf_flash.get(j, None), X_j)
            peak_prob[:, j]  = _get_probs(self.qrf_peak.get(j, None), X_j)
            acc_prob[:, j]   = _get_probs(self.qrf_acc.get(j, None), X_j)

        q_log_tensor = torch.clamp(torch.from_numpy(q_log).to(device), min=0.0)
        def prob_to_logits(p): return torch.log(torch.clamp(p, 1e-6, 1.0 - 1e-6) / (1.0 - torch.clamp(p, 1e-6, 1.0 - 1e-6)))
        return q_log_tensor, prob_to_logits(torch.from_numpy(flash_prob).to(device)), prob_to_logits(torch.from_numpy(peak_prob).to(device)), prob_to_logits(torch.from_numpy(acc_prob).to(device))

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()
    total_crps_log, total_crps_mm, total_brier_flash, total_brier_peak, total_brier_acc, nb = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    k50 = int(np.argmin(np.abs(np.array(list(quantiles), dtype=np.float32) - 0.5)))
    flash_p_list, flash_y_list, flash_m_list, peak_p_list, peak_y_list, peak_m_list, acc_p_list, acc_y_list, acc_m_list = [], [], [], [], [], [], [], [], []

    for x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc in loader:
        x, reg, y_log, my = x.to(device), reg.to(device), y_log.to(device), my.to(device)
        flash, mflash, peak, mpeak, acc, macc = flash.to(device), mflash.to(device), peak.to(device), mpeak.to(device), acc.to(device), macc.to(device)
        q_log_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        total_crps_log += crps_from_quantiles(q_log_hat, y_log, quantiles, mask=my).item()
        q_mm, y_mm = torch.expm1(q_log_hat).clamp_min(0.0), torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        p_flash, p_peak, p_acc = torch.sigmoid(flash_logits), torch.sigmoid(peak_logits), torch.sigmoid(acc_logits)
        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak += brier_score(p_peak, peak, mask=mpeak).item()
        total_brier_acc += brier_score(p_acc, acc, mask=macc).item()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1)); flash_y_list.append(flash.detach().cpu().reshape(-1)); flash_m_list.append(mflash.detach().cpu().reshape(-1))
        peak_p_list.append(p_peak.detach().cpu().reshape(-1)); peak_y_list.append(peak.detach().cpu().reshape(-1)); peak_m_list.append(mpeak.detach().cpu().reshape(-1))
        acc_p_list.append(p_acc.detach().cpu().reshape(-1)); acc_y_list.append(acc.detach().cpu().reshape(-1)); acc_m_list.append(macc.detach().cpu().reshape(-1))
        nb += 1

    flash_p, flash_y, flash_m = torch.cat(flash_p_list).numpy(), torch.cat(flash_y_list).numpy(), torch.cat(flash_m_list).numpy()
    peak_p, peak_y, peak_m = torch.cat(peak_p_list).numpy(), torch.cat(peak_y_list).numpy(), torch.cat(peak_m_list).numpy()
    acc_p, acc_y, acc_m = torch.cat(acc_p_list).numpy(), torch.cat(acc_y_list).numpy(), torch.cat(acc_m_list).numpy()

    return {
        "CRPS_log": total_crps_log / max(nb, 1), "CRPS_mm": total_crps_mm / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1), "Brier_peak": total_brier_peak / max(nb, 1), "Brier_acc": total_brier_acc / max(nb, 1),
        "Flash_metrics": event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds),
        "Peak24_metrics": event_metrics_binary(peak_p, peak_y, peak_m, thresholds=thresholds),
        "Acc24_metrics": event_metrics_binary(acc_p, acc_y, acc_m, thresholds=thresholds),
    }

# ============================================================
# 5) QRF Training Engine (FIXED)
# ============================================================
def sample_hparams_qrf(rng):
    # FIXED: Safely handle None for ints, and limit parameters to speed up CPU processing
    depth_choice = rng.choice([5, 10, 15, 20, None])
    max_depth = int(depth_choice) if depth_choice is not None else None

    feat_choice = rng.choice(["sqrt", "log2", 0.3, 0.5]) # Excluded None to speed up splits
    max_features = float(feat_choice) if isinstance(feat_choice, float) else feat_choice

    return {
        "n_estimators": int(rng.randint(50, 150)), # Shrunk from 300 to 150
        "max_depth": max_depth,
        "min_samples_split": int(rng.randint(5, 21)),
        "min_samples_leaf": int(rng.randint(5, 21)),
        "max_features": max_features
    }

def train_station_regressors_qrf(X_reg, y_reg, station_reg, lead_reg, params_base, num_stations, H_out):
    qrf_regressors = {}
    for j in range(num_stations):
        for h in range(H_out):
            mask_jh = (station_reg == j) & (lead_reg == h)
            if not np.any(mask_jh): continue
            X_jh, y_jh = X_reg[mask_jh], y_reg[mask_jh]
            if X_jh.shape[0] < 50: continue

            model = RandomForestQuantileRegressor(**params_base, random_state=42, n_jobs=-1)
            model.fit(X_jh, y_jh)
            qrf_regressors[(j, h)] = model
    return qrf_regressors

def train_station_classifiers_qrf(tab, num_stations, params_base):
    X_clf, station_clf = tab["X_clf"], tab["station_clf"]

    def train_one_target(y, m):
        models = {}
        for j in range(num_stations):
            mask_station = (station_clf == j)
            if not np.any(mask_station): continue
            X_j, y_j, m_j = X_clf[mask_station], y[mask_station], m[mask_station]
            valid = (m_j > 0.5)
            X_j, y_j = X_j[valid], y_j[valid]

            if X_j.shape[0] < 20 or len(np.unique(y_j)) < 2:
                const_val = y_j[0] if len(y_j) > 0 else 0
                model = DummyClassifier(strategy="constant", constant=int(const_val))
                model.fit(X_j if X_j.shape[0] > 0 else np.zeros((1, X_clf.shape[1])), [const_val])
                models[j] = model
                continue

            model = RandomForestClassifier(**params_base, random_state=42, n_jobs=-1)
            model.fit(X_j, y_j)
            models[j] = model
        return models

    return train_one_target(tab["flash_y"], tab["flash_m"]), train_one_target(tab["peak_y"], tab["peak_m"]), train_one_target(tab["acc_y"], tab["acc_m"])

# ============================================================
# 6) Master Tuning Script
# ============================================================
def tune_qrf_stationwise(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15, quantiles=(0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95), n_trials=10):
    prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(df_rain, T_in, H_out, train_frac, val_frac)
    train_end, val_end, T_total = splits
    ds_train, ds_val, ds_test = datasets

    tab_loader = DataLoader(ds_train, batch_size=256, shuffle=False, drop_last=False)
    val_loader = DataLoader(ds_val, batch_size=256, shuffle=False)
    N_stations = len(prep["stations"])

    print("Building tabular TRAIN data for QRF...")
    tab_train = build_tabular_from_loader(tab_loader)
    rng = np.random.RandomState(42)

    csv_path, json_path, pkl_path = os.path.join(TUNING_SAVE_DIR, "qrf_tuning_results.csv"), os.path.join(TUNING_SAVE_DIR, "best_trial.json"), os.path.join(TUNING_SAVE_DIR, "best_models.pkl")
    best_score, best_trial, trial_rows, start_trial = float("inf"), None, [], 1

    if os.path.exists(csv_path) and os.path.exists(json_path):
        print(f"\n[Info] Found existing tuning results in {TUNING_SAVE_DIR}. Reloading...")
        df_existing = pd.read_csv(csv_path)
        trial_rows = df_existing.to_dict('records')
        start_trial = len(trial_rows) + 1
        with open(json_path, "r") as f:
            best_trial = json.load(f)
            best_score = best_trial.get("val_CRPS_log", float("inf"))
        print(f"[Info] Resuming from trial {start_trial}. Current best CRPS_log: {best_score:.4f}")
    else:
        print("\n[Info] No previous results found. Starting fresh.")

    for trial in range(start_trial, n_trials + 1):
        print(f"\n[QRF Tuning] Trial {trial}/{n_trials}")
        hp = sample_hparams_qrf(rng)
        print("  Hyperparams:", hp)

        qrf_reg = train_station_regressors_qrf(tab_train["X_reg"], tab_train["y_reg"], tab_train["station_reg"], tab_train["lead_reg"], hp, N_stations, H_out)
        qrf_flash, qrf_peak, qrf_acc = train_station_classifiers_qrf(tab_train, N_stations, hp)

        model = StationWiseQRFBaseline(qrf_reg, qrf_flash, qrf_peak, qrf_acc, N_stations, H_out, quantiles).to(device)
        val_scores = evaluate(model, val_loader, quantiles)
        val_CRPS_log = float(val_scores["CRPS_log"])
        print(f"  -> val_CRPS_log = {val_CRPS_log:.4f}")

        row = {"trial": trial, "val_CRPS_log": val_CRPS_log}; row.update(hp)
        trial_rows.append(row)
        pd.DataFrame(trial_rows).to_csv(csv_path, index=False)

        if val_CRPS_log < best_score - 1e-6:
            best_score, best_trial = val_CRPS_log, row
            print(f"  -> New best trial (CRPS_log = {best_score:.4f})")
            with open(json_path, "w") as f: json.dump(best_trial, f, indent=2)
            joblib.dump({"qrf_reg": qrf_reg, "qrf_flash": qrf_flash, "qrf_peak": qrf_peak, "qrf_acc": qrf_acc}, pkl_path)

    return best_trial

# >>> EXECUTE TUNING HERE <<<
best_trial = tune_qrf_stationwise(df_rain=df_rain, n_trials=15)

Using device: cuda
Tuning directory: C:\Users\deepn\Documents\Rainfall_Training_Research_Lab\Baselines\QRF\Tuning_v1
Total stations: 34
Total unique times: 61360
N_stations: 34
Building tabular TRAIN data for QRF...


C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  X_reg: (11607236, 288) | y_reg: (11607236,)
  X_clf: (1450901, 288)

[Info] No previous results found. Starting fresh.

[QRF Tuning] Trial 1/10
  Hyperparams: {'n_estimators': 142, 'max_depth': 50, 'min_samples_split': 16, 'min_samples_leaf': 11, 'max_features': None}


C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
C:\Users\deepn\AppData\Local\Temp\ipykernel_15696\1069068501.py:331: RuntimeWarning: All-NaN slice encountered

  -> val_CRPS_log = 0.1263
  -> New best trial (CRPS_log = 0.1263)

[QRF Tuning] Trial 2/10


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

# **Model Training**

In [ ]:
# ============================================================
# STATION-WISE CATBOOST BASELINE - FINAL TRAINING PIPELINE
# ============================================================
#
# - Assumes df_rain (DataFrame) is ALREADY LOADED in memory.
# - Loads best hyperparameters from TUNING_SAVE_DIR/best_trial.json
# - Trains final CatBoost models on the Training Split
# - Evaluates performance on the held-out TEST SPLIT
# - Saves final models and metrics to FINAL_SAVE_DIR
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import joblib
import catboost as cb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility, Device, and Paths
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using PyTorch device:", device)

# CatBoost GPU check for massive speedup
CB_TASK_TYPE = "GPU" if torch.cuda.is_available() else "CPU"
print("CatBoost Task Type:", CB_TASK_TYPE)

TUNING_SAVE_DIR = r"/content/drive/MyDrive/Rainfall/Baselines/CatBoost/Tuning_v1"
FINAL_SAVE_DIR = r"/content/drive/MyDrive/Rainfall/Baselines/CatBoost/Final_Run_v1"
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)
print("Final artifacts will be saved to:", FINAL_SAVE_DIR)

# ============================================================
# 1) Core Utilities
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }

def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end

def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    train_flat = X_flat[:train_end*N]
    scaler = NaNIgnoringStandardScaler().fit(train_flat)
    X_scaled = scaler.transform(X_flat).reshape(T, N, F).astype(np.float32)
    return X_scaled, scaler

def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train = Y[:train_end]
    M_train = M[:train_end]
    N = Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)

    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0

    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        if len(vals) < 100:
            thr[j] = global_fallback
        else:
            thr[j] = np.nanpercentile(vals, q*100)
    return thr

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    times = np.sort(df["Datetime"].unique())
    T = len(times)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]
        m  = self.M_in[t-self.T_in+1:t+1]
        tf = self.time_feats[t-self.T_in+1:t+1]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)

        y  = self.Y_rain[t+1:t+1+self.H_out]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),
            torch.tensor(regime_id, dtype=torch.long),
            torch.from_numpy(y_log),
            torch.from_numpy(my),
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

# ============================================================
# 4) Evaluation Function
# ============================================================

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()

    total_crps_log = 0.0
    total_crps_mm  = 0.0
    total_brier_flash = 0.0
    total_brier_peak  = 0.0
    total_brier_acc   = 0.0
    nb = 0

    H_out = None
    sum_abs = None
    sum_sq  = None
    count   = None

    qs = np.array(list(quantiles), dtype=np.float32)
    k50 = int(np.argmin(np.abs(qs - 0.5)))

    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list,  peak_y_list,  peak_m_list  = [], [], []
    acc_p_list,   acc_y_list,   acc_m_list   = [], [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x = x.to(device)
        reg = reg.to(device)
        y_log = y_log.to(device)
        my = my.to(device)
        flash = flash.to(device); mflash = mflash.to(device)
        peak  = peak.to(device);  mpeak  = mpeak.to(device)
        acc   = acc.to(device);   macc   = macc.to(device)
        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()

        q_mm = torch.expm1(q_hat).clamp_min(0.0)
        y_mm = torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm  += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        p_flash = torch.sigmoid(flash_logits)
        p_peak  = torch.sigmoid(peak_logits)
        p_acc   = torch.sigmoid(acc_logits)

        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak  += brier_score(p_peak,  peak,  mask=mpeak ).item()
        total_brier_acc   += brier_score(p_acc,   acc,   mask=macc  ).item()

        nb += 1

        q50_log = q_hat[..., k50]
        q50_mm  = torch.expm1(q50_log).clamp_min(0.0)

        B, H, N = y_mm.shape
        if H_out is None:
            H_out = H
            sum_abs = torch.zeros(H_out, device="cpu")
            sum_sq  = torch.zeros(H_out, device="cpu")
            count   = torch.zeros(H_out, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h]  += (err**2).sum()
                count[h]   += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1))
        flash_y_list.append(flash.detach().cpu().reshape(-1))
        flash_m_list.append(mflash.detach().cpu().reshape(-1))

        peak_p_list.append(p_peak.detach().cpu().reshape(-1))
        peak_y_list.append(peak.detach().cpu().reshape(-1))
        peak_m_list.append(mpeak.detach().cpu().reshape(-1))

        acc_p_list.append(p_acc.detach().cpu().reshape(-1))
        acc_y_list.append(acc.detach().cpu().reshape(-1))
        acc_m_list.append(macc.detach().cpu().reshape(-1))

    out = {
        "CRPS_log":   total_crps_log / max(nb, 1),
        "CRPS_mm":    total_crps_mm  / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak":  total_brier_peak  / max(nb, 1),
        "Brier_acc":   total_brier_acc   / max(nb, 1),
    }

    maes, rmses, counts = [], [], []
    for h in range(H_out):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))
    out.update({"MAE_mm_by_lead": maes, "RMSE_mm_by_lead": rmses, "n_valid_by_lead": counts})

    flash_p = torch.cat(flash_p_list).numpy()
    flash_y = torch.cat(flash_y_list).numpy()
    flash_m = torch.cat(flash_m_list).numpy()

    peak_p  = torch.cat(peak_p_list).numpy()
    peak_y  = torch.cat(peak_y_list).numpy()
    peak_m  = torch.cat(peak_m_list).numpy()

    acc_p   = torch.cat(acc_p_list).numpy()
    acc_y   = torch.cat(acc_y_list).numpy()
    acc_m   = torch.cat(acc_m_list).numpy()

    out["Flash_metrics"]  = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    out["Peak24_metrics"] = event_metrics_binary(peak_p,  peak_y,  peak_m,  thresholds=thresholds)
    out["Acc24_metrics"]  = event_metrics_binary(acc_p,   acc_y,   acc_m,   thresholds=thresholds)

    return out

# ============================================================
# 5) Build Data Objects & Loaders
# ============================================================

def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac=train_frac, val_frac=val_frac)

    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)

    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)

    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24,
        prep["season_ids"], thr3h, thrAcc24,
        t_start=0, t_end=train_end, T_in=T_in, H_out=H_out
    )
    ds_val = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24,
        prep["season_ids"], thr3h, thrAcc24,
        t_start=train_end, t_end=val_end, T_in=T_in, H_out=H_out
    )
    ds_test = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"], Acc24, MaskAcc24,
        prep["season_ids"], thr3h, thrAcc24,
        t_start=val_end, t_end=T, T_in=T_in, H_out=H_out
    )

    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)

def make_loaders(ds_train, ds_val, ds_test, batch_size=256):
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# ============================================================
# 6) Flatten Loader to Tabular Data
# ============================================================

def build_tabular_from_loader(loader):
    X_reg_list, y_reg_list, station_reg_list, lead_reg_list, mask_reg_list = [], [], [], [], []
    X_clf_list, station_clf_list = [], []
    flash_y_list, flash_m_list = [], []
    peak_y_list,  peak_m_list  = [], []
    acc_y_list,   acc_m_list   = [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x_np = x[:, -1, :, :].numpy()
        y_mm = torch.expm1(y_log).clamp_min(0.0).numpy()
        my_np = my.numpy()

        flash_np = flash.numpy()
        mflash_np = mflash.numpy()
        peak_np  = peak.numpy()
        mpeak_np = mpeak.numpy()
        acc_np   = acc.numpy()
        macc_np  = macc.numpy()

        B, H, N = y_mm.shape

        for b in range(B):
            for h in range(H):
                mask_h = my_np[b, h, :] > 0.5
                if not mask_h.any(): continue
                X_bh = x_np[b]
                y_bh = y_mm[b, h, :]
                for j in range(N):
                    if mask_h[j]:
                        X_reg_list.append(X_bh[j])
                        y_reg_list.append(y_bh[j])
                        station_reg_list.append(j)
                        lead_reg_list.append(h)
                        mask_reg_list.append(1.0)

        for b in range(B):
            for j in range(N):
                if mflash_np[b, j] > 0.5:
                    X_clf_list.append(x_np[b, j])
                    station_clf_list.append(j)
                    flash_y_list.append(flash_np[b, j])
                    peak_y_list.append(peak_np[b, j])
                    acc_y_list.append(acc_np[b, j])
                    flash_m_list.append(mflash_np[b, j])
                    peak_m_list.append(mpeak_np[b, j])
                    acc_m_list.append(macc_np[b, j])

    return {
        "X_reg": np.array(X_reg_list, dtype=np.float32),
        "y_reg": np.array(y_reg_list, dtype=np.float32),
        "station_reg": np.array(station_reg_list, dtype=np.int64),
        "lead_reg": np.array(lead_reg_list, dtype=np.int64),
        "mask_reg": np.array(mask_reg_list, dtype=np.float32),

        "X_clf": np.array(X_clf_list, dtype=np.float32),
        "station_clf": np.array(station_clf_list, dtype=np.int64),
        "flash_y": np.array(flash_y_list, dtype=np.float32),
        "flash_m": np.array(flash_m_list, dtype=np.float32),
        "peak_y":  np.array(peak_y_list, dtype=np.float32),
        "peak_m":  np.array(peak_m_list, dtype=np.float32),
        "acc_y":   np.array(acc_y_list, dtype=np.float32),
        "acc_m":   np.array(acc_m_list, dtype=np.float32),
    }

# ============================================================
# 7) CatBoost PyTorch Wrapper
# ============================================================

class StationWiseCatBoostBaseline(nn.Module):
    def __init__(self, cb_regressors, cb_flash, cb_peak, cb_acc, num_stations, H_out, quantiles):
        super().__init__()
        self.cb_regressors = cb_regressors
        self.cb_flash = cb_flash
        self.cb_peak = cb_peak
        self.cb_acc = cb_acc
        self.N = num_stations
        self.H_out = H_out
        self.quantiles = list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        x_last = x[:, -1, :, :].detach().cpu().numpy()

        q_mm = np.zeros((B, self.H_out, N, self.K), dtype=np.float32)
        flash_prob = np.zeros((B, N), dtype=np.float32)
        peak_prob  = np.zeros((B, N), dtype=np.float32)
        acc_prob   = np.zeros((B, N), dtype=np.float32)

        for j in range(self.N):
            X_j = x_last[:, j, :]

            for k, q in enumerate(self.quantiles):
                model = self.cb_regressors.get((j, q), None)
                if model is None:
                    yhat = np.zeros(B, dtype=np.float32)
                else:
                    yhat = model.predict(X_j)
                q_mm[:, :, j, k] = yhat[:, None]

            m_flash = self.cb_flash.get(j, None)
            flash_prob[:, j] = m_flash.predict_proba(X_j)[:, 1] if m_flash else 0.0

            m_peak = self.cb_peak.get(j, None)
            peak_prob[:, j] = m_peak.predict_proba(X_j)[:, 1] if m_peak else 0.0

            m_acc = self.cb_acc.get(j, None)
            acc_prob[:, j] = m_acc.predict_proba(X_j)[:, 1] if m_acc else 0.0

        q_mm_tensor = torch.from_numpy(q_mm).to(device)
        q_log = torch.log1p(q_mm_tensor).clamp_min(0.0)

        def prob_to_logits(p):
            p = torch.clamp(p, 1e-6, 1.0 - 1e-6)
            return torch.log(p / (1.0 - p))

        flash_logits = prob_to_logits(torch.from_numpy(flash_prob).to(device))
        peak_logits  = prob_to_logits(torch.from_numpy(peak_prob).to(device))
        acc_logits   = prob_to_logits(torch.from_numpy(acc_prob).to(device))

        return q_log, flash_logits, peak_logits, acc_logits

# ============================================================
# 8) Final Training Execution Logic
# ============================================================

def train_cb_station_regressors(X_reg, y_reg, station_reg, params_base, quantiles, num_stations):
    cb_regressors = {}
    for j in range(num_stations):
        mask_j = (station_reg == j)
        if not np.any(mask_j): continue
        X_j, y_j = X_reg[mask_j], y_reg[mask_j]

        if X_j.shape[0] < 50: continue

        for q in quantiles:
            model = cb.CatBoostRegressor(
                loss_function=f'Quantile:alpha={q}',
                iterations=params_base["iterations"],
                depth=params_base["depth"],
                learning_rate=params_base["learning_rate"],
                l2_leaf_reg=params_base["l2_leaf_reg"],
                subsample=params_base["subsample"],
                bootstrap_type="Bernoulli",
                task_type=CB_TASK_TYPE,
                random_seed=42,
                verbose=0
            )
            model.fit(X_j, y_j)
            cb_regressors[(j, q)] = model
    return cb_regressors

def train_cb_station_classifiers(tab, num_stations, params_base):
    def train_one_target(y, m):
        models = {}
        for j in range(num_stations):
            mask_station = (tab["station_clf"] == j)
            if not np.any(mask_station): continue

            X_j, y_j, m_j = tab["X_clf"][mask_station], y[mask_station], m[mask_station]
            valid = (m_j > 0.5)
            X_j, y_j = X_j[valid], y_j[valid]

            if X_j.shape[0] < 20 or len(np.unique(y_j)) < 2: continue

            model = cb.CatBoostClassifier(
                loss_function='Logloss',
                iterations=params_base["iterations"],
                depth=params_base["depth"],
                learning_rate=params_base["learning_rate"],
                l2_leaf_reg=params_base["l2_leaf_reg"],
                subsample=params_base["subsample"],
                bootstrap_type="Bernoulli",
                task_type=CB_TASK_TYPE,
                random_seed=42,
                verbose=0
            )
            model.fit(X_j, y_j)
            models[j] = model
        return models

    cb_flash = train_one_target(tab["flash_y"], tab["flash_m"])
    cb_peak  = train_one_target(tab["peak_y"],  tab["peak_m"])
    cb_acc   = train_one_target(tab["acc_y"],   tab["acc_m"])
    return cb_flash, cb_peak, cb_acc

# ============================================================
# 9) Master Final Training Function
# ============================================================

def final_train_catboost(
    df_rain,
    best_trial_path,
    T_in=16,
    H_out=8,
    train_frac=0.7,
    val_frac=0.15,
    quantiles=(0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95),
):
    print(f"Loading best hyperparameters from: {best_trial_path}")
    with open(best_trial_path, "r") as f:
        best_hp = json.load(f)
    print("Hyperparameters:", json.dumps(best_hp, indent=2))

    # 1. Build Data Objects
    prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(
        df_rain, T_in=T_in, H_out=H_out, train_frac=train_frac, val_frac=val_frac,
    )
    train_end, val_end, T_total = splits
    ds_train, ds_val, ds_test = datasets

    train_loader, val_loader, test_loader = make_loaders(ds_train, ds_val, ds_test, batch_size=256)
    N_stations = len(prep["stations"])

    # 2. Build Tabular Data from the entire Training Split
    print("\nBuilding tabular TRAIN data from loader...")
    tab_train = build_tabular_from_loader(train_loader)

    # 3. Train Models
    print("\nTraining Final CatBoost Regressors (This may take a moment)...")
    cb_reg = train_cb_station_regressors(
        tab_train["X_reg"], tab_train["y_reg"], tab_train["station_reg"],
        best_hp, quantiles=quantiles, num_stations=N_stations,
    )

    print("Training Final CatBoost Classifiers...")
    cb_flash, cb_peak, cb_acc = train_cb_station_classifiers(
        tab_train, num_stations=N_stations, params_base=best_hp,
    )

    # 4. Save Final Models
    final_models = {
        "cb_reg": cb_reg,
        "cb_flash": cb_flash,
        "cb_peak": cb_peak,
        "cb_acc": cb_acc,
    }
    joblib.dump(final_models, os.path.join(FINAL_SAVE_DIR, "final_catboost_models.pkl"))
    print(f"Final models saved to {FINAL_SAVE_DIR}")

    # 5. Wrap inside PyTorch Module for unified evaluation
    model = StationWiseCatBoostBaseline(
        cb_regressors=cb_reg,
        cb_flash=cb_flash,
        cb_peak=cb_peak,
        cb_acc=cb_acc,
        num_stations=N_stations,
        H_out=H_out,
        quantiles=quantiles,
    ).to(device)

    # 6. Evaluate on Held-Out Test Set
    print("\n[EVALUATING ON HELD-OUT TEST SET]")
    test_scores = evaluate(model, test_loader, quantiles=quantiles)

    # 7. Save Metrics
    metrics_path = os.path.join(FINAL_SAVE_DIR, "test_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(test_scores, f, indent=2)

    print("\nFinal Test Metrics:")
    print(f"  Test CRPS (log): {test_scores['CRPS_log']:.4f}")
    print(f"  Test CRPS (mm):  {test_scores['CRPS_mm']:.4f}")
    print(f"  Test Brier (Flash): {test_scores['Brier_flash']:.4f}")
    print(f"  Test Brier (Peak):  {test_scores['Brier_peak']:.4f}")
    print(f"  Test Brier (Acc):   {test_scores['Brier_acc']:.4f}")

    return test_scores

# ============================================================
# 10) EXECUTE FINAL TRAINING
# ============================================================
# Uncomment the block below to run the pipeline!
#
# best_trial_path = os.path.join(TUNING_SAVE_DIR, "best_trial.json")
# final_test_metrics = final_train_catboost(
#     df_rain=df_rain,
#     best_trial_path=best_trial_path,
#     T_in=16,
#     H_out=8,
#     train_frac=0.7,
#     val_frac=0.15,
#     quantiles=(0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95),
# )

Using device: cpu
Tuning directory : C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1
Final save dir   : C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1

Loading best hyperparameters from: C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1\best_trial.json
Best tuning trial summary: {'trial': 2, 'val_CRPS_log': 0.13871812798283323, 'learning_rate': 0.013728250383906291, 'n_estimators': 763, 'num_leaves': 40, 'min_data_in_leaf': 150, 'feature_fraction': 0.5102922471479012, 'bagging_fraction': 0.9849549260809971, 'bagging_freq': 4, 'lambda_l2': 42.78743516759823}
Total stations: 34
Total unique times: 61360

Data split indices:
  Train: 0 → 42952
  Val:    42952 → 52156
  Test:   52156 → 61360
N_stations: 34

Building tabular TRAIN data...


C:\Users\pc\AppData\Local\Temp\ipykernel_9604\3569106490.py:338: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  TRAIN: X_reg: (11538022, 18) | y_reg: (11538022,)
         X_clf: (1442245, 18)

Building tabular VAL data (to include in final training)...
  VAL:   X_reg: (2494848, 18) | y_reg: (2494848,)
         X_clf: (311856, 18)

Merging TRAIN + VAL tabular datasets...
  FULL:  X_reg: (14032870, 18) | y_reg: (14032870,)
         X_clf: (1754101, 18)

[FINAL LightGBM] Training station-wise quantile regressors...

[FINAL LightGBM] Training station-wise classifiers (flash/peak/acc)...


# **Testing**

In [ ]:
# ============================================================
# STATION-WISE CATBOOST BASELINE - TESTING PIPELINE
# ============================================================
#
# - Assumes df_rain (DataFrame) is ALREADY LOADED in memory.
# - Assumes final training produced:
#       FINAL_SAVE_DIR/final_catboost_models.pkl
# - This script:
#       * Rebuilds preprocessing and splits (same as training)
#       * Uses SAVED scaler_mean/std, thr3h, thrAcc24, hyperparams
#       * Wraps the saved station-wise CatBoost models
#       * Evaluates on TEST split only (no retraining)
#       * Prints same metrics as other baselines:
#           - CRPS_log, CRPS_mm
#           - Brier_flash / Brier_peak / Brier_acc
#           - Event metrics (PR-AUC/ROC-AUC, POD/FAR/CSI)
#           - MAE/RMSE by lead
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import catboost as cb
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 0.5) Paths (EDIT THESE IF NEEDED)
# ------------------------------------------------------------
FINAL_SAVE_DIR = r"/content/drive/MyDrive/Rainfall/Baselines/CatBoost/Final_Run_v1"
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

FINAL_MODELS_PATH = os.path.join(FINAL_SAVE_DIR, "final_catboost_models.pkl")

print("Final models path :", FINAL_MODELS_PATH)

# ============================================================
# 1) Core utilities (must match training)
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        std_safe = np.where((self.std_ is not None) & (self.std_ < self.eps), 1.0, self.std_)
        return (X - self.mean_) / std_safe

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }

def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing: full grid, station/time features
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    # DR -> sin/cos
    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    print("Total stations:", N)

    times = np.sort(df["Datetime"].unique())
    T = len(times)
    print("Total unique times:", T)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    # time-of-day / month features
    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    # regime ids (season)
    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset (same structure as NN baselines)
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]     # [T_in,N,F]
        m  = self.M_in[t-self.T_in+1:t+1]         # [T_in,N,F]
        tf = self.time_feats[t-self.T_in+1:t+1]   # [T_in,N,4]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)  # [T_in,N,F+F+4]

        y  = self.Y_rain[t+1:t+1+self.H_out]       # [H_out,N]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        # flash at t+1
        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        # peak24
        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        # acc24
        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),                         # x
            torch.tensor(regime_id, dtype=torch.long),       # regime
            torch.from_numpy(y_log),                         # y_log
            torch.from_numpy(my),                            # my
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

def make_loaders(ds_test, batch_size=256):
    test_loader  = DataLoader(ds_test, batch_size=batch_size, shuffle=False)
    return test_loader

# ============================================================
# 4) Station-wise CatBoost model wrapper
# ============================================================

class StationWiseCatBoostBaseline(nn.Module):
    """
    CatBoost station-wise baseline:
      - One quantile regressor per (station, quantile).
      - One binary classifier per station for flash / peak / acc.
    Wrapped as nn.Module so `evaluate()` works unchanged.
    """
    def __init__(self,
                 cb_regressors,
                 cb_flash,
                 cb_peak,
                 cb_acc,
                 num_stations,
                 H_out,
                 quantiles):
        super().__init__()
        self.cb_regressors = cb_regressors
        self.cb_flash = cb_flash
        self.cb_peak = cb_peak
        self.cb_acc = cb_acc
        self.N = num_stations
        self.H_out = H_out
        self.quantiles = list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        assert N == self.N

        # last time step features
        x_last = x[:, -1, :, :].detach().cpu().numpy()  # [B,N,F]

        q_mm = np.zeros((B, self.H_out, N, self.K), dtype=np.float32)
        flash_prob = np.zeros((B, N), dtype=np.float32)
        peak_prob  = np.zeros((B, N), dtype=np.float32)
        acc_prob   = np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_last[:, j, :]  # [B,F]

            # Quantile regressors
            for k, q in enumerate(self.quantiles):
                model = self.cb_regressors.get((j, q), None)
                if model is None:
                    yhat = np.zeros(B, dtype=np.float32)
                else:
                    yhat = model.predict(X_j)
                q_mm[:, :, j, k] = yhat[:, None]  # same for all H_out

            # Binary classifiers: predict probabilities using predict_proba
            m_flash = self.cb_flash.get(j, None)
            if m_flash is None:
                flash_prob[:, j] = 0.0
            else:
                flash_prob[:, j] = m_flash.predict_proba(X_j)[:, 1]

            m_peak = self.cb_peak.get(j, None)
            if m_peak is None:
                peak_prob[:, j] = 0.0
            else:
                peak_prob[:, j] = m_peak.predict_proba(X_j)[:, 1]

            m_acc = self.cb_acc.get(j, None)
            if m_acc is None:
                acc_prob[:, j] = 0.0
            else:
                acc_prob[:, j] = m_acc.predict_proba(X_j)[:, 1]

        q_mm_tensor = torch.from_numpy(q_mm).to(device)
        q_log = torch.log1p(q_mm_tensor).clamp_min(0.0)

        def prob_to_logits(p):
            p = torch.clamp(p, 1e-6, 1.0 - 1e-6)
            return torch.log(p / (1.0 - p))

        flash_logits = prob_to_logits(torch.from_numpy(flash_prob).to(device))
        peak_logits  = prob_to_logits(torch.from_numpy(peak_prob).to(device))
        acc_logits   = prob_to_logits(torch.from_numpy(acc_prob).to(device))

        return q_log, flash_logits, peak_logits, acc_logits

# ============================================================
# 5) Evaluation (same as other baselines)
# ============================================================

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()

    total_crps_log = 0.0
    total_crps_mm  = 0.0
    total_brier_flash = 0.0
    total_brier_peak  = 0.0
    total_brier_acc   = 0.0
    nb = 0

    H_out = None
    sum_abs = None
    sum_sq  = None
    count   = None

    qs = np.array(list(quantiles), dtype=np.float32)
    k50 = int(np.argmin(np.abs(qs - 0.5)))

    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list,  peak_y_list,  peak_m_list  = [], [], []
    acc_p_list,   acc_y_list,   acc_m_list   = [], [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x = x.to(device)
        reg = reg.to(device)
        y_log = y_log.to(device)
        my = my.to(device)
        flash = flash.to(device); mflash = mflash.to(device)
        peak  = peak.to(device);  mpeak  = mpeak.to(device)
        acc   = acc.to(device);   macc   = macc.to(device)

        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        # CRPS (log space)
        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()

        # Convert to mm and compute CRPS
        q_mm = torch.expm1(q_hat).clamp_min(0.0)
        y_mm = torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm  += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        # Risk Brier
        p_flash = torch.sigmoid(flash_logits)
        p_peak  = torch.sigmoid(peak_logits)
        p_acc   = torch.sigmoid(acc_logits)

        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak  += brier_score(p_peak,  peak,  mask=mpeak ).item()
        total_brier_acc   += brier_score(p_acc,   acc,   mask=macc  ).item()

        nb += 1

        # Median quantile
        q50_log = q_hat[..., k50]
        q50_mm  = torch.expm1(q50_log).clamp_min(0.0)

        B, H, N = y_mm.shape
        if H_out is None:
            H_out = H
            sum_abs = torch.zeros(H_out, device="cpu")
            sum_sq  = torch.zeros(H_out, device="cpu")
            count   = torch.zeros(H_out, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h]  += (err**2).sum()
                count[h]   += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1))
        flash_y_list.append(flash.detach().cpu().reshape(-1))
        flash_m_list.append(mflash.detach().cpu().reshape(-1))

        peak_p_list.append(p_peak.detach().cpu().reshape(-1))
        peak_y_list.append(peak.detach().cpu().reshape(-1))
        peak_m_list.append(mpeak.detach().cpu().reshape(-1))

        acc_p_list.append(p_acc.detach().cpu().reshape(-1))
        acc_y_list.append(acc.detach().cpu().reshape(-1))
        acc_m_list.append(macc.detach().cpu().reshape(-1))

    out = {
        "CRPS_log":   total_crps_log / max(nb, 1),
        "CRPS_mm":    total_crps_mm  / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak":  total_brier_peak  / max(nb, 1),
        "Brier_acc":   total_brier_acc   / max(nb, 1),
    }

    # MAE / RMSE by lead
    maes, rmses, counts = [], [], []
    for h in range(H_out):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))
    out.update({"MAE_mm_by_lead": maes, "RMSE_mm_by_lead": rmses, "n_valid_by_lead": counts})

    # Event metrics
    flash_p = torch.cat(flash_p_list).numpy()
    flash_y = torch.cat(flash_y_list).numpy()
    flash_m = torch.cat(flash_m_list).numpy()

    peak_p  = torch.cat(peak_p_list).numpy()
    peak_y  = torch.cat(peak_y_list).numpy()
    peak_m  = torch.cat(peak_m_list).numpy()

    acc_p   = torch.cat(acc_p_list).numpy()
    acc_y   = torch.cat(acc_y_list).numpy()
    acc_m   = torch.cat(acc_m_list).numpy()

    out["Flash_metrics"]  = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    out["Peak24_metrics"] = event_metrics_binary(peak_p,  peak_y,  peak_m,  thresholds=thresholds)
    out["Acc24_metrics"]  = event_metrics_binary(acc_p,   acc_y,   acc_m,   thresholds=thresholds)

    return out

# ============================================================
# 6) LOAD FINAL MODELS + REBUILD TEST SET + EVALUATE
# ============================================================

print("\nLoading final CatBoost artifacts from:", FINAL_MODELS_PATH)
art = joblib.load(FINAL_MODELS_PATH)

quantiles   = tuple(art["quantiles"])
cb_reg      = art["cb_reg"]
cb_flash    = art["cb_flash"]
cb_peak     = art["cb_peak"]
cb_acc      = art["cb_acc"]
scaler_mean = np.array(art["scaler_mean"])
scaler_std  = np.array(art["scaler_std"])
thr3h       = np.array(art["thr3h"])
thrAcc24    = np.array(art["thrAcc24"])
T_in        = int(art["T_in"])
H_out       = int(art["H_out"])
train_frac  = float(art["train_frac"])
val_frac    = float(art["val_frac"])

print("Loaded quantiles   :", quantiles)
print("T_in, H_out        :", T_in, H_out)
print("train_frac, val_frac :", train_frac, val_frac)

# Rebuild preprocessing from df_rain
prep = prepare_full_grid(df_rain)
stations = prep["stations"]
N_stations = len(stations)

times = prep["times"]
T_total = len(times)

train_end, val_end = make_splits(T_total, train_frac=train_frac, val_frac=val_frac)
print("\nSplit indices (recomputed):")
print("  Train: 0 →", train_end)
print("  Val:   ", train_end, "→", val_end)
print("  Test:  ", val_end, "→", T_total)

# Rebuild scaler from saved stats (NO refit)
scaler = NaNIgnoringStandardScaler()
scaler.mean_ = scaler_mean
scaler.std_  = scaler_std

X_raw = prep["X_raw"]
T_, N_, F_in_raw = X_raw.shape
X_flat = X_raw.reshape(T_ * N_, F_in_raw)
X_scaled_flat = scaler.transform(X_flat)
X_scaled = X_scaled_flat.reshape(T_, N_, F_in_raw).astype(np.float32)

# Acc24 + MaskAcc24 recomputed (data-dependent only)
Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)

# Build TEST dataset only, using SAVED thresholds
ds_test = BanglaRainDataset(
    X_scaled, prep["M_in"], prep["time_feats"],
    prep["Y_rain"], prep["M_y"],
    Acc24, MaskAcc24,
    prep["season_ids"],
    thr3h, thrAcc24,
    t_start=val_end, t_end=T_total,
    T_in=T_in, H_out=H_out,
)

test_loader = make_loaders(ds_test, batch_size=256)
print("Test dataset size:", len(ds_test))

# Wrap CatBoost models in PyTorch-like module
model = StationWiseCatBoostBaseline(
    cb_regressors=cb_reg,
    cb_flash=cb_flash,
    cb_peak=cb_peak,
    cb_acc=cb_acc,
    num_stations=N_stations,
    H_out=H_out,
    quantiles=quantiles,
).to(device)

print("\nModel wrapper built. Evaluating on TEST split...")

with torch.no_grad():
    test_scores = evaluate(
        model,
        test_loader,
        quantiles=quantiles,
    )

# ============================================================
# 7) PRINT TEST METRICS
# ============================================================

print("\n========== CATBOOST STATION-WISE TEST METRICS ==========")
print(f"CRPS_log   : {test_scores['CRPS_log']:.4f}")
print(f"CRPS_mm    : {test_scores['CRPS_mm']:.4f}")
print(f"Brier_flash: {test_scores['Brier_flash']:.4f}")
print(f"Brier_peak : {test_scores['Brier_peak']:.4f}")
print(f"Brier_acc  : {test_scores['Brier_acc']:.4f}")

fm = test_scores["Flash_metrics"]
pm = test_scores["Peak24_metrics"]
am = test_scores["Acc24_metrics"]

print("\n[Flash 3h]  PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(fm["pr_auc"], fm["roc_auc"], fm["n_valid"]))
print("[Peak24]   PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(pm["pr_auc"], pm["roc_auc"], pm["n_valid"]))
print("[Acc24]    PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(am["pr_auc"], am["roc_auc"], am["n_valid"]))

print("\nMAE_mm_by_lead :", test_scores['MAE_mm_by_lead'])
print("RMSE_mm_by_lead:", test_scores['RMSE_mm_by_lead'])
print("n_valid_by_lead:", test_scores['n_valid_by_lead'])